# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Before testing the signals, I first looked at the distributions of the main numeric fields in the dataset.

The goal was to understand the typical values and identify whether any variables were highly skewed or had unusual ranges.

The main fields checked were:

- `impressions_90d`
- `clicks_90d`
- `content_age_days`
- `days_since_last_update`
- `ctr`
- `avg_position`

The distributions show that several metrics are strongly right-skewed, meaning that most pages have relatively small values while a smaller number of pages have very large values.

For example, `impressions_90d` has a median of **731**, while the mean is **5,200.37**, showing that a small number of high-volume pages pull the mean upward.

Similarly, `clicks_90d` has a median of **1** and a mean of **16.10**.

The `avg_position` field also needs special handling because **0 means no position data**, rather than an actual search ranking. There are **1,205 rows** with `avg_position == 0`.

These distributions are useful before the signal tests because they help identify skewed variables, sparse groups, and values that need to be interpreted carefully.

In [19]:
import pandas as pd
from pathlib import Path

# Use the repository's starter dataset; do not alter the data.
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
assert DATA_PATH.exists(), (
    "Starter CSV not found — run this notebook from the repository root."
)
df = pd.read_csv(DATA_PATH)

fields = [
    "impressions_90d",
    "clicks_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
]

missing_fields = sorted(set(fields) - set(df.columns))
assert not missing_fields, f"Missing expected columns: {missing_fields}"

# n is the non-missing count; missing is shown separately.
summary = pd.DataFrame({
    "n (non-missing)": df[fields].count(),
    "missing": df[fields].isna().sum(),
    "min": df[fields].min(),
    "median": df[fields].median(),
    "mean": df[fields].mean(),
    "max": df[fields].max(),
})
summary.index.name = "field"

quantiles = df[fields].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T
quantiles.columns = ["p25", "p50", "p75", "p90", "p95", "p99"]
quantiles.index.name = "field"

print("Distribution summary")
print(summary.round(2).to_string())
print("\nQuantiles (use the upper percentiles to spot long or heavy tails)")
print(quantiles.round(2).to_string())

zero_position_rows = (df["avg_position"] == 0).sum()
print(
    f"avg_position == 0: {zero_position_rows:,} rows "
    "(0 means no position data, not rank zero)."
)

Distribution summary
                        n (non-missing)  missing   min  median     mean       max
field                                                                            
impressions_90d                   30000        0   1.0  731.00  5200.37  517715.0
clicks_90d                        30000        0   0.0    1.00    16.10    4178.0
content_age_days                  30000        0  90.0  236.00   256.17     564.0
days_since_last_update            30000        0   1.0   20.00    46.10     373.0
ctr                               30000        0   0.0    0.07     0.51     100.0
avg_position                      30000        0   0.0   10.80    16.34     245.0

Quantiles (use the upper percentiles to spot long or heavy tails)
                          p25     p50      p75       p90       p95       p99
field                                                                       
impressions_90d          81.0  731.00  3615.25  12136.40  22996.50  73505.83
clicks_90d               

## 2. Signal tests

I tested several signals to see whether they showed a clear relationship with observed decline.

The purpose of these tests was not to prove that one signal causes decline, but to check whether the patterns in the data support the assumptions behind potential action signals.

I looked at:

1. **Staleness → Engagement**

   This test checks whether pages that have not been updated for a longer time show lower engagement compared with fresher pages. The idea is that older or less recently updated content may need more attention, but the result should be treated as an observed pattern rather than proof that staleness causes lower engagement.

2. **Search position → CTR**

   This test checks whether pages appearing lower in search results receive a lower click-through rate (CTR). The purpose is to see whether search visibility is reflected in user clicks. A clear pattern here could support using search position as a useful signal when prioritizing pages for review.

3. **Impressions → Decline**

   This test checks whether pages with different levels of search impressions have different observed decline rates. This helps determine whether page traffic volume is useful when identifying pages that may deserve attention.

4. **CTR → Decline**

   This test checks whether pages with different click-through rates have different observed decline rates. The goal is to see whether CTR provides any useful signal about which pages are more likely to show a decline.

Each test was evaluated using the observed data in the corresponding buckets.

The verdicts were:

- **Staleness → Engagement:** MIXED

- **Search position → CTR:** CONFIRMED

- **Impressions → Decline:** MIXED

- **CTR → Decline:** MIXED

The results show that some signals have useful patterns, but no single signal gives a consistently strong relationship with decline across all groups.

Therefore, these signals should be treated as supporting evidence rather than automatic decision rules.

In [37]:
# Signal test 1 - Staleness / refresh signal

staleness_df = df.copy()

# Create freshness buckets
staleness_df["bucket"] = pd.cut(
    staleness_df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

# Calculate observed decline rate and page count for each bucket
staleness_test = (
    staleness_df
    .groupby("bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        decline_count=("is_declining_label", "sum"),
        decline_rate=("is_declining_label", "mean")
    )
    .reset_index()
)

# Convert decline rate to percentage
staleness_test["decline_rate"] = staleness_test["decline_rate"] * 100
print("------------------------------------------")
print("Signal test 1 - Staleness / refresh signal")
print("------------------------------------------")
display(staleness_test)


# Signal test 2 - Volume / quick-win signal
print("------------------------------------------")
print("Signal test 2 - Volume / quick-win signal")
print("------------------------------------------")
volume_df = df.copy()

# Create impression-volume buckets
volume_df["bucket"] = pd.cut(
    volume_df["impressions_90d"],
    bins=[0, 299, 2999, 29999, float("inf")],
    labels=[
        "1-299 impressions",
        "300-2,999 impressions",
        "3,000-29,999 impressions",
        "30,000+ impressions"
    ]
)

# Calculate observed decline rate and page count for each bucket
volume_test = (
    volume_df
    .groupby("bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        decline_count=("is_declining_label", "sum"),
        decline_rate=("is_declining_label", "mean")
    )
    .reset_index()
)

# Convert decline rate to percentage
volume_test["decline_rate"] = volume_test["decline_rate"] * 100

display(volume_test)


# Signal test 3 - Search-ranking signal
print("------------------------------------------")
print(" Signal test 3 - Search-ranking signal")
print("------------------------------------------")

position_df = df.copy()

# Create search-ranking buckets
position_df["bucket"] = pd.cut(
    position_df["avg_position"],
    bins=[-1, 3, 10, 20, 50, float("inf")],
    labels=[
        "top 3",
        "page 1",
        "striking distance",
        "pages 3-5",
        "deep"
    ]
)

# Add "no position data" as a category
position_df["bucket"] = position_df["bucket"].cat.add_categories(
    ["no position data"]
)

# Keep rows with no position data as a separate bucket
position_df.loc[
    position_df["avg_position"] == 0,
    "bucket"
] = "no position data"

# Calculate observed decline rate and page count for each bucket
position_test = (
    position_df
    .groupby("bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        decline_count=("is_declining_label", "sum"),
        decline_rate=("is_declining_label", "mean")
    )
    .reset_index()
)

# Convert decline rate to percentage
position_test["decline_rate"] = position_test["decline_rate"] * 100

display(position_test)

# Signal test 4 - Search volume vs observed decline
print("------------------------------------------")
print(" Signal test 4 - Search volume vs observed decline")
print("------------------------------------------")

volume_df = df.copy()

# Create impression-volume buckets
volume_df["impression_tier"] = pd.cut(
    volume_df["impressions_90d"],
    bins=[-1, 100, 1000, 10000, float("inf")],
    labels=["0-100", "101-1,000", "1,001-10,000", "10,000+"]
)

# Calculate observed decline rate and page count for each bucket
volume_test = (
    volume_df
    .groupby("impression_tier", observed=False)
    .agg(
        n=("content_id", "count"),
        decline_rate=("is_declining_label", "mean")
    )
    .reset_index()
)

# Convert decline rate to percentage
volume_test["decline_rate"] = volume_test["decline_rate"] * 100

display(volume_test)

------------------------------------------
Signal test 1 - Staleness / refresh signal
------------------------------------------


,bucket,n,decline_count,decline_rate
0,0-30 days,20480,10473,51.137695
1,31-90 days,175,103,58.857143
2,91-180 days,9171,5604,61.105659
3,181+ days,174,82,47.126437


------------------------------------------
Signal test 2 - Volume / quick-win signal
------------------------------------------


,bucket,n,decline_count,decline_rate
0,1-299 impressions,11248,5106,45.394737
1,"300-2,999 impressions",10469,6435,61.467189
2,"3,000-29,999 impressions",7205,4223,58.612075
3,"30,000+ impressions",1078,498,46.196660


------------------------------------------
 Signal test 3 - Search-ranking signal
------------------------------------------


,bucket,n,decline_count,decline_rate
0,top 3,1141,568,49.780894
1,page 1,11842,6743,56.941395
2,striking distance,7273,4433,60.951464
3,pages 3-5,7225,4059,56.179931
4,deep,1314,451,34.322679
5,no position data,1205,8,0.663900


------------------------------------------
 Signal test 4 - Search volume vs observed decline
------------------------------------------


,impression_tier,n,decline_rate
0,0-100,8006,38.920809
1,"101-1,000",8485,60.282852
2,"1,001-10,000",9907,62.026850
3,"10,000+",3602,52.359800


## 3. The flag-linked test

### Staleness / refresh flag assumption

I tested the assumption behind FlyRank's staleness / refresh flag: pages that have not been updated for a longer time may be more likely to decline.

For this test, I divided the pages into two groups:

- **Not stale:** fewer than 91 days since the last update
- **Stale:** 91 days or more since the last update

### What did the data show?

The **Not stale** group had an observed decline rate of **51.20%**, while the **Stale** group had an observed decline rate of **60.85%**.

This means the stale group had a **9.65 percentage-point higher** observed decline rate than the not-stale group.

### What does this tell us?

The result provides **descriptive support** for the staleness / refresh assumption because the stale pages had a higher observed decline rate.

However, this does not mean that being stale causes a page to decline. It only shows that the two groups had different observed decline rates in this dataset.

### Human-review limitation

A page may not have been updated recently because the content is already accurate, evergreen, or performing well. Therefore, staleness should be treated as a signal for **human review**, rather than an automatic reason to refresh a page.

In [21]:
# Use the documented starter-data label only for this descriptive comparison.
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

staleness_flag = df["days_since_last_update"].ge(91)
flag_table = (
    df.assign(
        stale_status=staleness_flag.map(
            {False: "Not stale (<91 days)", True: "Stale (91+ days)"}
        )
    )
    .groupby("stale_status")["is_declining_label"]
    .agg(
        n="size",
        decline_count="sum",
        decline_rate="mean"
    )
    .reindex(["Not stale (<91 days)", "Stale (91+ days)"])
    .reset_index()
)

flag_table["decline_rate"] = (flag_table["decline_rate"] * 100).round(2)

not_stale_rate = flag_table.loc[
    flag_table["stale_status"].eq("Not stale (<91 days)"), "decline_rate"
].iloc[0]
stale_rate = flag_table.loc[
    flag_table["stale_status"].eq("Stale (91+ days)"), "decline_rate"
].iloc[0]
rate_difference_pp = stale_rate - not_stale_rate

print("Staleness / refresh flag comparison")
print(flag_table.to_string(index=False))
print(f"\nStale minus not-stale decline rate: {rate_difference_pp:+.2f} percentage points")

Staleness / refresh flag comparison
        stale_status     n  decline_count  decline_rate
Not stale (<91 days) 20655          10576         51.20
    Stale (91+ days)  9345           5686         60.85

Stale minus not-stale decline rate: +9.65 percentage points


## 4. What this means in practice

From these tests, I found that no single signal gives a clear and consistent pattern across all buckets. So I would not use any one signal by itself to decide that a page needs action.

The staleness signal was interesting. In the flag-linked test, pages that had not been updated for 91 days or more had a 60.85% decline rate, compared with 51.20% for pages updated more recently. This gives some support to the idea behind the refresh flag.

However, this does not mean that old content causes a decline. There can be other reasons why a page has not been updated, such as the content already being accurate or evergreen.

The position and CTR tests also show that search-performance signals can be useful, but not every signal has a consistent relationship with decline. Because of this, I think these signals are better used together as supporting information for human review, rather than as automatic decisions.

The main takeaway is that a signal can show a useful pattern without being strong enough to become a decision rule on its own. I will use this as a guide when building the transparent baseline action score in ML-07.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.